# 16 — Training a CNN for Image Classification

In the previous notebook, we learned the foundations of Convolutional Neural Networks:

- Image tensor structure
- Convolution
- Kernels
- Feature maps
- `nn.Conv2d`
- Stride
- Padding
- Pooling
- Flattening
- CNN shape reasoning

Now we will connect those ideas into a complete image-classification pipeline.

We will build a small synthetic image dataset so that every part of the pipeline can run directly in Google Colab without downloading external data.

Then we will train a CNN using:

- `Dataset`
- `DataLoader`
- Image normalization
- Data augmentation
- `CrossEntropyLoss`
- Adam optimizer
- Training and validation loops
- Accuracy
- Confusion matrix
- Best-model checkpointing

Finally, we will discuss how to adapt the exact same structure to real ultrasound images.

## In this notebook, we will study:

1. Creating an image-classification dataset
2. CNN input pipelines
3. Image normalization
4. Data augmentation
5. Train / validation / test loaders
6. Building a practical CNN
7. `CrossEntropyLoss`
8. CNN training loop
9. Validation loop
10. Accuracy
11. Confusion matrix
12. Saving the best CNN
13. Test-set evaluation
14. Overfitting in CNNs
15. Improving CNN performance
16. Preparing the pipeline for real ultrasound data
17. Common CNN training mistakes
18. Practice exercises

## Main Goal

By the end of this notebook, you should understand the complete image-classification pipeline:

$$
\boxed{
\text{Images}
\rightarrow
\text{Transforms}
\rightarrow
\text{DataLoader}
\rightarrow
\text{CNN}
\rightarrow
\text{Logits}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
}
$$

and the evaluation pipeline:

$$
\boxed{
\text{Validation/Test Images}
\rightarrow
\text{CNN}
\rightarrow
\text{Predictions}
\rightarrow
\text{Metrics}
}
$$


In [ ]:
import copy
import math
import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import transforms

print("PyTorch version:", torch.__version__)


# 1. The Image-Classification Pipeline

A practical CNN project usually contains these major components:

$$
\begin{array}{|c|c|}
\hline
\textbf{Component} & \textbf{Purpose} \\
\hline
Dataset & \text{Returns one image and one target} \\
\hline
Transforms & \text{Preprocess or augment an image} \\
\hline
DataLoader & \text{Builds mini-batches} \\
\hline
CNN & \text{Produces class logits} \\
\hline
Loss & \text{Measures prediction error} \\
\hline
Optimizer & \text{Updates model parameters} \\
\hline
Validation & \text{Measures generalization} \\
\hline
Checkpoint & \text{Stores the best model} \\
\hline
\end{array}
$$

We will build each part carefully.


# 2. Our Synthetic Image Problem

We will create grayscale images with three classes.

Each image has shape:

$$
\boxed{
(1,\ 64,\ 64)
}
$$

The three classes will contain different bright structures:

- Class 0 — mostly vertical structure
- Class 1 — mostly horizontal structure
- Class 2 — cross-like structure

Noise and small position changes will make the problem less trivial.

This dataset is only for learning the CNN pipeline.


# 3. Why Use Synthetic Images First?

A synthetic dataset lets us focus on:

- Tensor shapes
- CNN architecture
- Data loading
- Augmentation
- Training
- Validation
- Metrics

without being distracted by:

- File formats
- DICOM handling
- Patient metadata
- Corrupted files
- Complex folder structures

Once the pipeline is understood, we can replace the synthetic dataset with real image files.


# 4. Creating One Synthetic Image

Let's create one 64×64 grayscale image.


In [ ]:
torch.manual_seed(42)

image = torch.zeros(
    1,
    64,
    64
)

image[
    :,
    :,
    30:34
] = 1.0

print(
    "Image shape:",
    image.shape
)


# 5. Visualizing One Image

For plotting, we remove the channel dimension:

$$
(1,\ 64,\ 64)
\rightarrow
(64,\ 64)
$$


In [ ]:
plt.figure(figsize=(4, 4))

plt.imshow(
    image.squeeze(0).numpy(),
    cmap="gray"
)

plt.title("Example Synthetic Image")
plt.axis("off")
plt.show()


# 6. Building a Synthetic-Image Generator

We will create a function that:

1. Starts with a dark image
2. Adds a class-specific structure
3. Randomly changes its position slightly
4. Adds Gaussian noise
5. Clips values into a stable range


In [ ]:
def make_synthetic_image(
    class_index,
    image_size=64,
    noise_std=0.15
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    horizontal_shift = torch.randint(
        low=-8,
        high=9,
        size=(1,)
    ).item()

    vertical_shift = torch.randint(
        low=-8,
        high=9,
        size=(1,)
    ).item()

    x_center = center + horizontal_shift
    y_center = center + vertical_shift

    thickness = 4

    x_start = max(
        0,
        x_center - thickness // 2
    )

    x_end = min(
        image_size,
        x_start + thickness
    )

    y_start = max(
        0,
        y_center - thickness // 2
    )

    y_end = min(
        image_size,
        y_start + thickness
    )

    if class_index == 0:
        image[
            :,
            :,
            x_start:x_end
        ] = 1.0

    elif class_index == 1:
        image[
            :,
            y_start:y_end,
            :
        ] = 1.0

    elif class_index == 2:
        image[
            :,
            :,
            x_start:x_end
        ] = 1.0

        image[
            :,
            y_start:y_end,
            :
        ] = 1.0

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    noise = torch.randn_like(
        image
    ) * noise_std

    image = image + noise

    image = image.clamp(
        0.0,
        1.0
    )

    return image


# 7. Visualizing All Three Classes


In [ ]:
torch.manual_seed(7)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

for class_index in range(3):
    example = make_synthetic_image(
        class_index
    )

    axes[class_index].imshow(
        example.squeeze(0).numpy(),
        cmap="gray"
    )

    axes[class_index].set_title(
        f"Class {class_index}"
    )

    axes[class_index].axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 8. Creating the Full Dataset

Let's create:

$$
300
$$

images per class.

Total:

$$
900
$$

images.


In [ ]:
torch.manual_seed(42)

samples_per_class = 300

all_images = []
all_targets = []

for class_index in range(3):
    for _ in range(
        samples_per_class
    ):
        all_images.append(
            make_synthetic_image(
                class_index
            )
        )

        all_targets.append(
            class_index
        )

images = torch.stack(
    all_images
)

targets = torch.tensor(
    all_targets,
    dtype=torch.long
)

print(
    "Images:",
    images.shape
)

print(
    "Targets:",
    targets.shape
)


The full dataset shape is:

$$
\boxed{
(900,\ 1,\ 64,\ 64)
}
$$

Target shape:

$$
\boxed{
(900)
}
$$


# 9. Inspecting the Class Distribution

Before training a classifier, inspect class counts.


In [ ]:
class_counts = torch.bincount(
    targets
)

print(
    "Class counts:",
    class_counts
)


Our synthetic dataset is balanced.

Real datasets often are not.

Class imbalance can affect:

- Accuracy interpretation
- Loss weighting
- Sampling strategy
- Threshold selection
- Evaluation metrics


# 10. Shuffling Before Splitting

The images were created class-by-class.

We should shuffle before splitting.


In [ ]:
torch.manual_seed(42)

permutation = torch.randperm(
    len(images)
)

images = images[
    permutation
]

targets = targets[
    permutation
]

print(
    "First 20 targets:",
    targets[:20]
)


# 11. Train / Validation / Test Split

We will use:

$$
70\%
$$

for training,

$$
15\%
$$

for validation,

and:

$$
15\%
$$

for final testing.

The test set will not be used for model selection.


In [ ]:
num_samples = len(
    images
)

train_end = int(
    0.70
    * num_samples
)

val_end = int(
    0.85
    * num_samples
)

train_images = images[
    :train_end
]

train_targets = targets[
    :train_end
]

val_images = images[
    train_end:val_end
]

val_targets = targets[
    train_end:val_end
]

test_images = images[
    val_end:
]

test_targets = targets[
    val_end:
]

print(
    "Train:",
    train_images.shape,
    train_targets.shape
)

print(
    "Validation:",
    val_images.shape,
    val_targets.shape
)

print(
    "Test:",
    test_images.shape,
    test_targets.shape
)


# 12. Why Split Before Computing Normalization Statistics?

Normalization parameters should be calculated from the training data only.

If we use validation or test images to calculate preprocessing statistics, information from evaluation data leaks into the training pipeline.

So the correct order is:

$$
\boxed{
\text{Split}
\rightarrow
\text{Compute Training Statistics}
\rightarrow
\text{Normalize All Splits Using Those Statistics}
}
$$


# 13. Image Normalization

A common normalization is:

$$
\boxed{
x_{norm}
=
\frac{x-\mu}{\sigma}
}
$$

where:

- $\mu$ = training-set mean
- $\sigma$ = training-set standard deviation

For grayscale images, we need one mean and one standard deviation.


In [ ]:
train_mean = train_images.mean(
    dim=(0, 2, 3)
)

train_std = train_images.std(
    dim=(0, 2, 3)
)

print(
    "Training mean:",
    train_mean
)

print(
    "Training std:",
    train_std
)


# 14. Why Normalize Images?

Normalization can help optimization by putting pixel values into a more consistent numerical range.

It can:

- Improve optimization stability
- Make training more predictable
- Help different datasets use a consistent scale

But normalization does not magically fix poor data quality.


# 15. Creating an Evaluation Transform

Validation and test transforms should be deterministic.

We will apply only normalization.


In [ ]:
eval_transform = transforms.Compose([
    transforms.Normalize(
        mean=train_mean.tolist(),
        std=train_std.tolist()
    )
])

print(eval_transform)


# 16. Data Augmentation

Data augmentation creates altered versions of training samples while preserving the label.

Common examples include:

- Horizontal flips
- Small rotations
- Translations
- Crops
- Brightness changes
- Contrast changes
- Noise

The goal is to expose the model to useful variability.


# 17. Training Transform

For our synthetic shapes, horizontal and vertical flips preserve class meaning.

We will use:

- Random horizontal flip
- Random vertical flip
- Small random rotation
- Normalization


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.Normalize(
        mean=train_mean.tolist(),
        std=train_std.tolist()
    )
])

print(train_transform)


# 18. Important Augmentation Rule

An augmentation is valid only if it preserves the correct label.

This is extremely important.

For example:

- A horizontal flip may be valid for one task
- The same flip may be anatomically invalid for another task

In medical imaging, augmentation should respect:

- Anatomy
- Acquisition orientation
- Clinical meaning
- Label semantics

Never use augmentations only because they are popular.


# 19. Custom Tensor Image Dataset

We will create a dataset that stores tensors and optionally applies a transform.


In [ ]:
class ImageTensorDataset(Dataset):
    def __init__(
        self,
        images,
        targets,
        transform=None
    ):
        self.images = images
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(
            self.images
        )

    def __getitem__(
        self,
        index
    ):
        image = self.images[
            index
        ]

        target = self.targets[
            index
        ]

        if self.transform is not None:
            image = self.transform(
                image
            )

        return (
            image,
            target
        )


# 20. Creating the Three Datasets

Training uses random augmentation.

Validation and test use deterministic preprocessing.


In [ ]:
train_dataset = ImageTensorDataset(
    train_images,
    train_targets,
    transform=train_transform
)

val_dataset = ImageTensorDataset(
    val_images,
    val_targets,
    transform=eval_transform
)

test_dataset = ImageTensorDataset(
    test_images,
    test_targets,
    transform=eval_transform
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)


# 21. Inspecting One Dataset Sample


In [ ]:
sample_image, sample_target = (
    train_dataset[0]
)

print(
    "Image shape:",
    sample_image.shape
)

print(
    "Target:",
    sample_target
)

print(
    "Image dtype:",
    sample_image.dtype
)

print(
    "Target dtype:",
    sample_target.dtype
)


For `CrossEntropyLoss`:

Target dtype should be:

`torch.long`

Image dtype should normally be floating point.


# 22. Visualizing Augmentation

Because the training transform is random, repeated access to the same index can produce different versions.


In [ ]:
fig, axes = plt.subplots(
    1,
    4,
    figsize=(12, 3)
)

for index in range(4):
    augmented_image, target = (
        train_dataset[0]
    )

    axes[index].imshow(
        augmented_image.squeeze(0).numpy(),
        cmap="gray"
    )

    axes[index].set_title(
        f"Target {target.item()}"
    )

    axes[index].axis(
        "off"
    )

plt.tight_layout()
plt.show()


The displayed images are normalized, so pixel values may no longer lie between 0 and 1.

That is expected.


# 23. DataLoader Setup

We will use:

- Shuffle for training
- No shuffle for validation/test
- `num_workers=0` for notebook simplicity


In [ ]:
batch_size = 32

generator = (
    torch.Generator()
    .manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


# 24. Inspecting One Batch


In [ ]:
batch_images, batch_targets = next(
    iter(train_loader)
)

print(
    "Image batch:",
    batch_images.shape
)

print(
    "Target batch:",
    batch_targets.shape
)

print(
    "Image dtype:",
    batch_images.dtype
)

print(
    "Target dtype:",
    batch_targets.dtype
)


The image batch should have shape:

$$
\boxed{
(N,\ 1,\ 64,\ 64)
}
$$

For a full batch:

$$
\boxed{
(32,\ 1,\ 64,\ 64)
}
$$

Targets:

$$
\boxed{
(32)
}
$$


# 25. Building a Practical Small CNN

We will build three convolution blocks.

Architecture:

$$
1
\rightarrow
16
\rightarrow
32
\rightarrow
64
$$

Each block uses:

$$
Conv
\rightarrow
ReLU
\rightarrow
MaxPool
$$

Then:

$$
Flatten
\rightarrow
Linear
\rightarrow
ReLU
\rightarrow
Dropout
\rightarrow
Linear
$$


In [ ]:
class PracticalCNN(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                64 * 8 * 8,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                p=0.3
            ),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):
        x = self.features(
            x
        )

        x = self.classifier(
            x
        )

        return x

model = PracticalCNN(
    num_classes=3
)

print(model)


# 26. CNN Shape Reasoning

Input:

$$
(N,\ 1,\ 64,\ 64)
$$

After Block 1:

$$
(N,\ 16,\ 32,\ 32)
$$

After Block 2:

$$
(N,\ 32,\ 16,\ 16)
$$

After Block 3:

$$
(N,\ 64,\ 8,\ 8)
$$

Flatten:

$$
(N,\ 4096)
$$

Hidden linear layer:

$$
(N,\ 128)
$$

Output logits:

$$
\boxed{
(N,\ 3)
}
$$


# 27. Verifying the Forward Shape


In [ ]:
dummy_batch = torch.randn(
    8,
    1,
    64,
    64
)

dummy_output = model(
    dummy_batch
)

print(
    "Input:",
    dummy_batch.shape
)

print(
    "Output:",
    dummy_output.shape
)


# 28. Inspecting Every Feature Shape


In [ ]:
x = torch.randn(
    2,
    1,
    64,
    64
)

print(
    "Input:",
    x.shape
)

for index, layer in enumerate(
    model.features
):
    x = layer(
        x
    )

    print(
        f"After features[{index}] "
        f"{layer.__class__.__name__}:",
        x.shape
    )


# 29. Parameter Count


In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)


# 30. Where Are the Parameters?

Convolution layers contain:

- Kernel weights
- Biases

Linear layers contain:

- Matrix weights
- Biases

ReLU, MaxPool, Flatten, and Dropout do not contain trainable weights.


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        tuple(parameter.shape),
        "numel =",
        parameter.numel()
    )


# 31. Loss Function

We have:

$$
3
$$

mutually exclusive classes.

The model outputs:

$$
3
$$

raw logits.

Targets contain class indices:

$$
0,\ 1,\ 2
$$

Therefore we use:

`nn.CrossEntropyLoss()`


In [ ]:
criterion = nn.CrossEntropyLoss()

print(criterion)


# 32. Why No Softmax Before the Loss?

`CrossEntropyLoss` expects raw logits.

Correct:

```python
logits = model(images)

loss = criterion(
    logits,
    targets
)
```

Do not do:

```python
softmax(logits)
```

before the training loss.

Softmax can be applied later if probabilities are needed for interpretation.


# 33. Optimizer

We will use Adam with a small amount of weight decay.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(optimizer)


# 34. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    device
)

print(
    "Using device:",
    device
)


# 35. One CNN Training Batch


In [ ]:
model.train()

batch_images, batch_targets = next(
    iter(train_loader)
)

batch_images = batch_images.to(
    device
)

batch_targets = batch_targets.to(
    device
)

optimizer.zero_grad()

logits = model(
    batch_images
)

loss = criterion(
    logits,
    batch_targets
)

loss.backward()

optimizer.step()

print(
    "Logits:",
    logits.shape
)

print(
    "Loss:",
    loss.item()
)


# 36. Batch Accuracy


In [ ]:
with torch.no_grad():
    predictions = logits.argmax(
        dim=1
    )

    batch_accuracy = (
        predictions
        == batch_targets
    ).float().mean()

print(
    "Batch accuracy:",
    batch_accuracy.item()
)


# 37. Reusable CNN Training Function


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, targets in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        total_correct += (
            predictions
            == targets
        ).sum().item()

        total_samples += (
            batch_size
        )

    epoch_loss = (
        total_loss
        / total_samples
    )

    epoch_accuracy = (
        total_correct
        / total_samples
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 38. Reusable Validation Function


In [ ]:
def evaluate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            total_correct += (
                predictions
                == targets
            ).sum().item()

            total_samples += (
                batch_size
            )

    epoch_loss = (
        total_loss
        / total_samples
    )

    epoch_accuracy = (
        total_correct
        / total_samples
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 39. Why Validation Uses `eval()` and `no_grad()`

`model.eval()` changes the behavior of layers such as:

- Dropout
- BatchNorm

`torch.no_grad()` disables gradient tracking.

Validation should not call:

- `loss.backward()`
- `optimizer.step()`

Validation measures performance.

Training changes parameters.


# 40. Full CNN Training Loop

We will:

1. Train one epoch
2. Validate one epoch
3. Store metrics
4. Save the best validation model


In [ ]:
torch.manual_seed(42)

model = PracticalCNN(
    num_classes=3
).to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs = 15

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")

best_state = copy.deepcopy(
    model.state_dict()
)

best_epoch = 0

for epoch in range(epochs):
    train_loss, train_accuracy = (
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )
    )

    val_loss, val_accuracy = (
        evaluate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )
    )

    history[
        "train_loss"
    ].append(
        train_loss
    )

    history[
        "train_accuracy"
    ].append(
        train_accuracy
    )

    history[
        "val_loss"
    ].append(
        val_loss
    )

    history[
        "val_accuracy"
    ].append(
        val_accuracy
    )

    if val_loss < best_val_loss:
        best_val_loss = (
            val_loss
        )

        best_epoch = (
            epoch + 1
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.3f}"
    )

print(
    "Best epoch:",
    best_epoch
)


# 41. Restoring the Best CNN

The last epoch is not always the best epoch.

Restore the best validation checkpoint.


In [ ]:
model.load_state_dict(
    best_state
)

print(
    "Best validation loss:",
    best_val_loss
)


# 42. Plotting Training and Validation Loss


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Training loss"
)

plt.plot(
    history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("CNN Training vs Validation Loss")
plt.legend()
plt.show()


# 43. Plotting Training and Validation Accuracy


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Training accuracy"
)

plt.plot(
    history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training vs Validation Accuracy")
plt.legend()
plt.show()


# 44. Interpreting the Curves

A useful pattern is:

- Training loss decreases
- Validation loss decreases
- Training accuracy increases
- Validation accuracy increases

Possible overfitting pattern:

- Training loss keeps decreasing
- Training accuracy becomes very high
- Validation loss begins increasing
- Validation accuracy stops improving

The validation curve should guide model selection.


# 45. Saving the Best CNN to Disk

After restoring `best_state`, save:

`model.state_dict()`


In [ ]:
best_model_path = (
    "best_practical_cnn.pth"
)

torch.save(
    model.state_dict(),
    best_model_path
)

print(
    "Saved:",
    best_model_path
)


# 46. Loading the Saved CNN

The architecture must be recreated before loading the state dictionary.


In [ ]:
loaded_model = PracticalCNN(
    num_classes=3
).to(
    device
)

loaded_state = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True
)

loaded_model.load_state_dict(
    loaded_state
)

loaded_model.eval()

print(
    "Model loaded successfully."
)


# 47. Validation Performance of the Best CNN


In [ ]:
best_val_loss, best_val_accuracy = (
    evaluate_one_epoch(
        loaded_model,
        val_loader,
        criterion,
        device
    )
)

print(
    "Validation loss:",
    best_val_loss
)

print(
    "Validation accuracy:",
    best_val_accuracy
)


# 48. Final Test Evaluation

The test set should be used only after model selection is complete.

We do not:

- Tune architecture on test results
- Tune learning rate on test results
- Select checkpoint based on test results


In [ ]:
test_loss, test_accuracy = (
    evaluate_one_epoch(
        loaded_model,
        test_loader,
        criterion,
        device
    )
)

print(
    "Test loss:",
    test_loss
)

print(
    "Test accuracy:",
    test_accuracy
)


# 49. Collecting Predictions

To build a confusion matrix, collect:

- True targets
- Predicted classes
- Probabilities


In [ ]:
def collect_predictions(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(
                device
            )

            logits = model(
                images
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = logits.argmax(
                dim=1
            )

            all_targets.append(
                targets.cpu()
            )

            all_predictions.append(
                predictions.cpu()
            )

            all_probabilities.append(
                probabilities.cpu()
            )

    return (
        torch.cat(
            all_targets
        ),
        torch.cat(
            all_predictions
        ),
        torch.cat(
            all_probabilities
        )
    )

test_true, test_pred, test_prob = (
    collect_predictions(
        loaded_model,
        test_loader,
        device
    )
)

print(
    "Targets:",
    test_true.shape
)

print(
    "Predictions:",
    test_pred.shape
)

print(
    "Probabilities:",
    test_prob.shape
)


# 50. Confusion Matrix Intuition

For three classes:

$$
\begin{array}{c|c|c|c}
 & Pred\ 0 & Pred\ 1 & Pred\ 2 \\
\hline
True\ 0 & ? & ? & ? \\
\hline
True\ 1 & ? & ? & ? \\
\hline
True\ 2 & ? & ? & ? \\
\end{array}
$$

Rows:

> True classes

Columns:

> Predicted classes

Diagonal:

> Correct predictions

Off-diagonal entries:

> Misclassifications


# 51. Building the Confusion Matrix Manually


In [ ]:
def confusion_matrix_torch(
    targets,
    predictions,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for true_label, predicted_label in zip(
        targets,
        predictions
    ):
        matrix[
            true_label.long(),
            predicted_label.long()
        ] += 1

    return matrix

confusion = confusion_matrix_torch(
    test_true,
    test_pred,
    num_classes=3
)

print(confusion)


# 52. Visualizing the Confusion Matrix


In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(
    confusion.numpy()
)

plt.title(
    "Test Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)

plt.xticks(
    range(3)
)

plt.yticks(
    range(3)
)

for row in range(3):
    for col in range(3):
        plt.text(
            col,
            row,
            int(
                confusion[
                    row,
                    col
                ]
            ),
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()


# 53. Per-Class Accuracy

Overall accuracy can hide class-specific weaknesses.

For class $i$:

$$
class\ accuracy_i
=
\frac{
confusion[i,i]
}{
\sum_j confusion[i,j]
}
$$


In [ ]:
for class_index in range(3):
    class_total = (
        confusion[
            class_index
        ].sum().item()
    )

    class_correct = (
        confusion[
            class_index,
            class_index
        ].item()
    )

    class_accuracy = (
        class_correct
        / class_total
        if class_total > 0
        else 0.0
    )

    print(
        f"Class {class_index}: "
        f"{class_accuracy:.3f}"
    )


# 54. Looking at Individual Predictions

It is useful to inspect examples that the model predicts correctly and incorrectly.


In [ ]:
example_images = test_images[
    :8
]

example_targets = test_targets[
    :8
]

normalized_examples = torch.stack([
    eval_transform(
        image
    )
    for image in example_images
])

loaded_model.eval()

with torch.no_grad():
    logits = loaded_model(
        normalized_examples.to(
            device
        )
    )

    predictions = logits.argmax(
        dim=1
    ).cpu()

print(
    "Targets:",
    example_targets
)

print(
    "Predictions:",
    predictions
)


In [ ]:
fig, axes = plt.subplots(
    2,
    4,
    figsize=(10, 5)
)

for index, axis in enumerate(
    axes.flat
):
    axis.imshow(
        example_images[
            index,
            0
        ].numpy(),
        cmap="gray"
    )

    axis.set_title(
        f"T={example_targets[index].item()} "
        f"P={predictions[index].item()}"
    )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 55. Accuracy Is Not Enough for Every Problem

For balanced synthetic data, accuracy is easy to interpret.

For real medical datasets, especially imbalanced ones, also consider:

- Sensitivity
- Specificity
- Precision
- Recall
- F1 score
- AUROC
- AUPRC
- Calibration

The clinically important metric depends on the task.


# 56. Overfitting in CNNs

CNNs can contain many parameters.

A model can overfit when it learns training-specific details instead of generalizable patterns.

Common signs:

- Training loss keeps decreasing
- Validation loss begins increasing
- Training accuracy is much higher than validation accuracy


# 57. Why CNNs Overfit

Possible reasons include:

- Too little training data
- Model too large
- Training too long
- Weak regularization
- Duplicate or highly correlated samples
- Distribution shift
- Data leakage
- Labels contain noise

Overfitting is not solved by one universal trick.


# 58. Improving CNN Performance — Better Data

Before increasing model complexity, inspect the data.

Questions to ask:

1. Are labels correct?
2. Are images consistently preprocessed?
3. Are train and validation distributions comparable?
4. Are there duplicates?
5. Is there patient leakage?
6. Are classes imbalanced?
7. Are acquisition devices different?
8. Is image quality consistent?

A clean dataset can matter more than another architectural layer.


# 59. Improving CNN Performance — Data Augmentation

Training augmentation can improve robustness when transformations are realistic.

Examples may include:

- Small rotation
- Translation
- Mild scaling
- Contrast variation
- Intensity variation
- Noise

But every augmentation should preserve label meaning.


# 60. Improving CNN Performance — Weight Decay

Weight decay can reduce overfitting.

Example:

```python
torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
```

We already used a small value in this notebook.


# 61. Improving CNN Performance — Dropout

Dropout can regularize the fully connected part of a CNN.

We used:

```python
nn.Dropout(
    p=0.3
)
```

During training, some activations are randomly dropped.

During evaluation:

`model.eval()`

disables dropout randomness.


# 62. Improving CNN Performance — Learning Rate

A learning rate that is too large can make training unstable.

A learning rate that is too small can make training extremely slow.

Learning rate should be tuned using validation performance.


# 63. Improving CNN Performance — Model Capacity

Possible changes include:

- More channels
- Fewer channels
- More convolution blocks
- Smaller classifier
- Larger classifier

More parameters are not automatically better.

Use controlled experiments.


# 64. Improving CNN Performance — Early Stopping

If validation performance stops improving, continued training may only increase overfitting.

Early stopping can:

- Reduce unnecessary training
- Restore the best validation checkpoint

Monitor validation data, not the test set.


# 65. Improving CNN Performance — Learning-Rate Scheduling

Later, we will study learning-rate schedulers.

A scheduler can reduce the learning rate during training.

Examples include:

- Step schedules
- Plateau-based reduction
- Cosine schedules

The optimizer learning rate does not need to remain constant.


# 66. Improving CNN Performance — Transfer Learning

For many real image problems, especially with limited data, pretrained CNNs can outperform a small network trained completely from scratch.

Transfer learning uses a model pretrained on a large dataset and adapts it to a new task.

We will study transfer learning in a later notebook.


# 67. Preparing for Real Ultrasound Data

The overall pipeline remains:

$$
\boxed{
File
\rightarrow
Load
\rightarrow
Preprocess
\rightarrow
Augment
\rightarrow
Tensor
\rightarrow
CNN
}
$$

But real ultrasound data introduces additional issues:

- File formats
- Device differences
- Image depth
- Cropping
- Text overlays
- Measurement markers
- Patient-level splitting
- Variable image sizes
- Multiple images per patient
- Class imbalance


# 68. Example Real Ultrasound Dataset Structure

A custom dataset may store:

- Image file path
- Label
- Patient ID
- Study ID
- Device/site information

Conceptually:

```python
class UltrasoundDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):
        row = self.dataframe.iloc[
            index
        ]

        image = load_image(
            row["path"]
        )

        label = row[
            "label"
        ]

        if self.transform:
            image = self.transform(
                image
            )

        return image, label
```

The exact loader depends on the file format.


# 69. Ultrasound Shape Convention

If preprocessing produces one grayscale channel:

$$
\boxed{
(C,\ H,\ W)
=
(1,\ 256,\ 256)
}
$$

A batch becomes:

$$
\boxed{
(N,\ 1,\ 256,\ 256)
}
$$

The first CNN layer should then use:

```python
nn.Conv2d(
    in_channels=1,
    ...
)
```


# 70. What If Ultrasound Files Are Stored as RGB?

Some ultrasound images may be stored as three-channel files even though they visually look grayscale.

Then the raw tensor may be:

$$
(3,\ H,\ W)
$$

You have two possible strategies:

1. Keep three channels and use `in_channels=3`
2. Convert deliberately to one grayscale channel and use `in_channels=1`

Choose consistently.

Do not silently mix both formats.


# 71. Patient-Level Splitting

This is especially important for medical imaging.

Suppose one patient contributes:

$$
20
$$

ultrasound images.

If images are randomly split individually, some images from that patient might enter training while other images from the same patient enter validation.

The model may then partially recognize patient-specific or acquisition-specific patterns.

That can make validation performance look artificially strong.

Whenever appropriate, split by:

- Patient
- Subject
- Study
- Acquisition session

rather than individual image.


# 72. Site and Device Leakage

Ultrasound appearance can vary across:

- Hospitals
- Scanner manufacturers
- Probes
- Acquisition protocols
- Operators

A model may accidentally learn site/device signals instead of pathology.

Useful evaluation designs may include:

- Site-stratified splits
- External-site testing
- Device subgroup analysis

The correct design depends on the research question.


# 73. Ultrasound Normalization

Possible normalization approaches include:

- Scaling known intensity range
- Per-image standardization
- Dataset-level standardization
- Robust percentile scaling

There is no universal best method.

Whatever you use:

- Fit training-derived statistics only on training data
- Apply the same deterministic rule to validation/test
- Document the preprocessing exactly


# 74. Ultrasound Augmentation Caution

Potentially reasonable augmentations may include:

- Mild rotations
- Small translations
- Modest scaling
- Mild contrast/intensity changes
- Carefully chosen noise

But appropriateness depends on:

- Anatomy
- Orientation
- Probe convention
- Diagnostic target

For example, a vertical flip may be unrealistic for many ultrasound tasks.

Use domain-valid augmentation rather than generic augmentation by habit.


# 75. Avoiding Text and Marker Leakage

Medical images may contain:

- Labels
- Measurements
- Arrows
- Calipers
- Machine text
- Hospital identifiers

If those elements correlate with the target, a model may exploit them.

Possible safeguards include:

- Cropping overlays
- Masking metadata regions
- Reviewing saliency/attention qualitatively
- Evaluating across sites/devices

Always inspect what the model could potentially use.


# 76. Common Mistake — Augmenting Validation Data Randomly

Training:

> Random augmentation can be useful.

Validation/Test:

> Usually deterministic preprocessing.

Random validation augmentation can make metrics unstable and harder to compare.


# 77. Common Mistake — Computing Mean and Std on the Full Dataset

This leaks validation/test information into preprocessing.

Correct:

$$
\boxed{
\mu_{train},
\sigma_{train}
}
$$

are calculated from training data only.

Then they are reused for all splits.


# 78. Common Mistake — Wrong Input Shape

CNN input should usually be:

$$
(N,\ C,\ H,\ W)
$$

not:

$$
(N,\ H,\ W,\ C)
$$

Always inspect:

```python
images.shape
```

before training.


# 79. Common Mistake — Wrong `in_channels`

If:

$$
images.shape=(32,\ 1,\ 64,\ 64)
$$

then the first convolution normally needs:

```python
in_channels=1
```


# 80. Common Mistake — Wrong Classifier Size

Our current model assumes input size:

$$
64\times64
$$

because:

$$
64
\rightarrow
32
\rightarrow
16
\rightarrow
8
$$

after three pooling layers.

If input resolution changes, the flattened dimension changes.

Always re-check the classifier input features.


# 81. Common Mistake — Applying Softmax Before `CrossEntropyLoss`

Use:

```python
logits = model(images)

loss = criterion(
    logits,
    targets
)
```

Do not apply softmax before `CrossEntropyLoss`.


# 82. Common Mistake — Training on Validation Data

Validation should not call:

- `loss.backward()`
- `optimizer.step()`

Validation measures generalization.


# 83. Common Mistake — Selecting the Best Model on Test Accuracy

The test set should remain untouched until the final evaluation.

Use validation metrics for:

- Hyperparameter tuning
- Architecture selection
- Checkpoint selection
- Early stopping


# 84. Common Mistake — Looking Only at Overall Accuracy

Especially for medical data, also inspect:

- Per-class performance
- Confusion matrix
- Sensitivity
- Specificity
- Precision
- Recall
- AUROC
- AUPRC

The most important metric depends on the clinical objective.


# 85. CNN Training Debugging Checklist

Before a long run, verify:

1. One raw image
2. One transformed image
3. Image shape
4. Image dtype
5. Target shape
6. Target dtype
7. One DataLoader batch
8. Class distribution
9. Model input channels
10. Model output logits
11. Loss function
12. Gradient existence
13. Parameter updates
14. Train vs validation curves
15. Checkpoint logic


In [ ]:
debug_images, debug_targets = next(
    iter(train_loader)
)

print(
    "Images:",
    debug_images.shape
)

print(
    "Targets:",
    debug_targets.shape
)

debug_images = debug_images.to(
    device
)

debug_targets = debug_targets.to(
    device
)

model.train()

debug_logits = model(
    debug_images
)

debug_loss = criterion(
    debug_logits,
    debug_targets
)

print(
    "Logits:",
    debug_logits.shape
)

print(
    "Loss:",
    debug_loss.item()
)


# 86. Checking Gradient Flow


In [ ]:
optimizer.zero_grad()

debug_logits = model(
    debug_images
)

debug_loss = criterion(
    debug_logits,
    debug_targets
)

debug_loss.backward()

for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        print(
            name,
            "| grad norm:",
            parameter.grad.norm().item()
        )


# 87. Checking Whether Parameters Change


In [ ]:
before = (
    model.features[0]
    .weight
    .detach()
    .clone()
)

optimizer.step()

after = (
    model.features[0]
    .weight
    .detach()
    .clone()
)

print(
    "First convolution changed:",
    not torch.equal(
        before,
        after
    )
)


# 88. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Create 100 grayscale images with shape:

$$
(1,\ 32,\ 32)
$$

and random class labels for 4 classes.

## Exercise 2

Create a `Dataset` that optionally applies a transform.

## Exercise 3

Calculate training-set mean and standard deviation.

## Exercise 4

Create:

- Training transform with augmentation
- Validation transform without random augmentation

## Exercise 5

Create train, validation, and test loaders.

## Exercise 6

Build a CNN:

$$
1
\rightarrow
8
\rightarrow
16
\rightarrow
32
$$

using:

- Conv
- ReLU
- MaxPool

## Exercise 7

For input:

$$
(32,\ 1,\ 64,\ 64)
$$

find the shape after three pooling layers.

## Exercise 8

Add a classifier for 5 classes.

## Exercise 9

Write a complete training and validation loop.

## Exercise 10

Save the model with the lowest validation loss.


# 89. Conceptual Challenges

Answer without running code first.

## Challenge 1

Why should normalization statistics come only from training data?

## Challenge 2

Why should augmentation normally be random only for training data?

## Challenge 3

For 6 classes, what should the final CNN output shape be for batch size 32?

## Challenge 4

Why does `CrossEntropyLoss` require raw logits instead of softmax probabilities?

## Challenge 5

Why might patient-level splitting be necessary for ultrasound images?

## Challenge 6

Why can a high training accuracy and lower validation accuracy indicate overfitting?

## Challenge 7

Why can the last epoch be worse than an earlier checkpoint?

## Challenge 8

Why might a model trained on one ultrasound machine perform poorly on another machine?

## Challenge 9

Why can image overlays cause leakage?

## Challenge 10

Why is test-set performance not supposed to guide model development?


# 90. Exercise Solutions


In [ ]:
# Exercise 1
exercise_images = torch.randn(
    100,
    1,
    32,
    32
)

exercise_targets = torch.randint(
    0,
    4,
    (100,)
)

print(
    "Exercise 1:",
    exercise_images.shape,
    exercise_targets.shape
)

# Exercise 2
class ExerciseImageDataset(Dataset):
    def __init__(
        self,
        images,
        targets,
        transform=None
    ):
        self.images = images
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(
            self.images
        )

    def __getitem__(
        self,
        index
    ):
        image = self.images[
            index
        ]

        target = self.targets[
            index
        ]

        if self.transform is not None:
            image = self.transform(
                image
            )

        return image, target

# Exercise 3
exercise_mean = exercise_images.mean(
    dim=(0, 2, 3)
)

exercise_std = exercise_images.std(
    dim=(0, 2, 3)
)

print(
    "Exercise 3 mean:",
    exercise_mean
)

print(
    "Exercise 3 std:",
    exercise_std
)

# Exercise 4
exercise_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.Normalize(
        exercise_mean.tolist(),
        exercise_std.tolist()
    )
])

exercise_eval_transform = transforms.Compose([
    transforms.Normalize(
        exercise_mean.tolist(),
        exercise_std.tolist()
    )
])

# Exercise 5
exercise_train_dataset = (
    ExerciseImageDataset(
        exercise_images[:70],
        exercise_targets[:70],
        transform=exercise_train_transform
    )
)

exercise_val_dataset = (
    ExerciseImageDataset(
        exercise_images[70:85],
        exercise_targets[70:85],
        transform=exercise_eval_transform
    )
)

exercise_test_dataset = (
    ExerciseImageDataset(
        exercise_images[85:],
        exercise_targets[85:],
        transform=exercise_eval_transform
    )
)

exercise_train_loader = DataLoader(
    exercise_train_dataset,
    batch_size=16,
    shuffle=True
)

exercise_val_loader = DataLoader(
    exercise_val_dataset,
    batch_size=16,
    shuffle=False
)

exercise_test_loader = DataLoader(
    exercise_test_dataset,
    batch_size=16,
    shuffle=False
)

print(
    "Exercise 5 complete."
)

# Exercise 6
exercise_cnn = nn.Sequential(
    nn.Conv2d(
        1,
        8,
        3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        8,
        16,
        3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        16,
        32,
        3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

print(
    "Exercise 6:"
)

print(
    exercise_cnn
)

# Exercise 7
exercise_input = torch.randn(
    32,
    1,
    64,
    64
)

exercise_features = exercise_cnn(
    exercise_input
)

print(
    "Exercise 7:",
    exercise_features.shape
)

# Exercise 8
exercise_flattened = torch.flatten(
    exercise_features,
    start_dim=1
)

exercise_classifier = nn.Linear(
    exercise_flattened.shape[1],
    5
)

exercise_logits = exercise_classifier(
    exercise_flattened
)

print(
    "Exercise 8:",
    exercise_logits.shape
)


# 91. Key Takeaways

In this notebook, we learned:

- How to create an image-classification dataset
- CNN image input pipelines
- Training-only normalization statistics
- Data augmentation
- Deterministic validation/test preprocessing
- Custom image datasets
- Train / validation / test loaders
- Building a practical CNN
- CNN shape reasoning
- `CrossEntropyLoss`
- Adam optimization
- CNN training loops
- Validation loops
- Accuracy
- Best-model checkpointing
- Test evaluation
- Confusion matrices
- Per-class accuracy
- Overfitting
- Dropout
- Weight decay
- Learning-rate tuning
- Transfer-learning motivation
- Real ultrasound pipeline considerations
- Patient-level splitting
- Device/site leakage
- Overlay leakage
- CNN debugging

The complete training pipeline is:

$$
\boxed{
Training\ Images
\rightarrow
Augmentation
\rightarrow
Normalization
\rightarrow
DataLoader
\rightarrow
CNN
\rightarrow
Logits
\rightarrow
Loss
\rightarrow
Backward
\rightarrow
Optimizer
}
$$

The evaluation pipeline is:

$$
\boxed{
Validation/Test\ Images
\rightarrow
Deterministic\ Preprocessing
\rightarrow
CNN
\rightarrow
Predictions
\rightarrow
Metrics
}
$$


# 92. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why should image data be split before normalization statistics are computed?
2. What is data augmentation?
3. Why should augmentation preserve label meaning?
4. Why are validation transforms usually deterministic?
5. What input shape does a PyTorch CNN usually expect?
6. Why must the first convolution's `in_channels` match the image tensor?
7. What should the final output shape be for a 3-class classifier?
8. Why should raw logits be passed to `CrossEntropyLoss`?
9. What happens during one CNN training batch?
10. Why is `model.eval()` used for validation?
11. Why is `torch.no_grad()` used for validation?
12. How is classification accuracy calculated?
13. What does a confusion matrix show?
14. Why can per-class accuracy be more informative than overall accuracy?
15. What is overfitting?
16. How can augmentation reduce overfitting?
17. How can weight decay help?
18. Why save the best validation checkpoint?
19. Why should the test set remain untouched until final evaluation?
20. Why is patient-level splitting important for many medical-imaging datasets?
21. Why can site/device differences hurt ultrasound generalization?
22. Why must ultrasound augmentations be clinically plausible?


# Next Notebook

# 17 — Regularization, Initialization, and Stable Training

In the next notebook, we will study:

- What is regularization?
- Overfitting vs underfitting
- Dropout in depth
- Weight decay
- L1 vs L2 intuition
- Data augmentation as regularization
- Early stopping
- Weight initialization
- Xavier initialization
- Kaiming initialization
- Why initialization matters
- Vanishing and exploding activations
- Gradient clipping
- Stable training habits
- Comparing regularization strategies
